In [0]:
# Publish Daniel's versioned raw tables in the team's existing catalog and schema.
# Standalone copies allow the team to choose a Silver input independently.
release_rows = {
    "facilities": 10088,
    "india_post_pincode_directory": 165627,
    "nfhs_5_district_health_indicators": 706,
}
for dataset, expected in release_rows.items():
    source = f"workspace.default.bronze_{dataset}"
    target = f"{source}_daniel"
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {target} USING DELTA
        AS SELECT * FROM {source} VERSION AS OF 0
    """)
    history = spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").first()
    version = int(history["version"])
    published = spark.sql(f"SELECT * FROM {target} VERSION AS OF {version}")
    accepted = spark.sql(f"SELECT * FROM {source} VERSION AS OF 0")
    if published.dtypes != accepted.dtypes:
        raise ValueError(f"Review existing target schema: {target}")
    rows = published.count()
    missing = accepted.exceptAll(published).count()
    extra = published.exceptAll(accepted).count()
    print(f"{target} | version={version} | rows={rows} | missing={missing} | extra={extra}")
    if rows != expected or missing or extra:
        raise ValueError(f"Review target contents before choosing it for Silver: {target}")

workspace.default.bronze_facilities_daniel | version=0 | rows=10088 | missing=0 | extra=0
workspace.default.bronze_india_post_pincode_directory_daniel | version=0 | rows=165627 | missing=0 | extra=0
workspace.default.bronze_nfhs_5_district_health_indicators_daniel | version=0 | rows=706 | missing=0 | extra=0


In [0]:
# PIN directory — raw geography at the agreed Bronze version
pin_table = "workspace.default.bronze_india_post_pincode_directory VERSION AS OF 0"
pin_fields = spark.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT_IF(pincode IS NULL) AS missing_pin,
           COUNT_IF(pincode IS NOT NULL AND
                    CAST(pincode AS STRING) NOT RLIKE '^[1-9][0-9]{{5}}$') AS malformed_pin,
           COUNT_IF(NULLIF(TRIM(officename), '') IS NULL) AS missing_office,
           COUNT_IF(NULLIF(TRIM(statename), '') IS NULL) AS missing_state,
           COUNT_IF(NULLIF(TRIM(district), '') IS NULL) AS missing_district
    FROM {pin_table}
""").first().asDict()
pin_keys = spark.sql(f"""
    WITH pins AS (
        SELECT pincode, COUNT(*) AS office_rows,
               COUNT(DISTINCT NULLIF(LOWER(TRIM(statename)), ''),
                              NULLIF(LOWER(TRIM(district)), '')) AS places
        FROM {pin_table} WHERE pincode IS NOT NULL GROUP BY pincode
    )
    SELECT COUNT(*) AS distinct_nonnull_pins,
           COUNT_IF(office_rows > 1) AS pins_with_multiple_rows,
           COUNT_IF(places = 0) AS pins_without_complete_location,
           COUNT_IF(places > 1) AS pins_with_multiple_state_district_pairs
    FROM pins
""").first().asDict()
print("PIN fields", pin_fields)
print("PIN groups", pin_keys)
assert pin_fields["rows"] == 165627, "Review the PIN input version and row count"

PIN fields {'rows': 165627, 'missing_pin': 0, 'malformed_pin': 0, 'missing_office': 0, 'missing_state': 0, 'missing_district': 0}
PIN groups {'distinct_nonnull_pins': 19586, 'pins_with_multiple_rows': 17443, 'pins_without_complete_location': 0, 'pins_with_multiple_state_district_pairs': 1478}


In [0]:
# NFHS-5 — geographic keys and indicator storage at the agreed Bronze version
from collections import Counter
from pyspark.sql import functions as F

nfhs = spark.sql("""
    SELECT * FROM workspace.default.bronze_nfhs_5_district_health_indicators VERSION AS OF 0
""")
keys = nfhs.selectExpr(
    "NULLIF(LOWER(TRIM(state_ut)), '') AS state",
    "NULLIF(LOWER(TRIM(district_name)), '') AS district"
).groupBy("state", "district").count()
geography = keys.selectExpr(
    "SUM(count) AS rows",
    "SUM(IF(state IS NULL OR district IS NULL, count, 0)) AS incomplete_location_rows",
    "COUNT_IF(state IS NOT NULL AND district IS NOT NULL) AS complete_location_pairs",
    "COUNT_IF(state IS NOT NULL AND district IS NOT NULL AND count > 1) AS repeated_complete_pairs"
).first().asDict()
measures = [(c, t) for c, t in nfhs.dtypes if c not in ("state_ut", "district_name")]
text_measures = [c for c, t in measures if t == "string"]
print("NFHS geography", geography)
print("NFHS measure columns by stored type", dict(Counter(t for _, t in measures)))
if text_measures:
    cells = nfhs.select(F.explode(F.array(*[F.col(c) for c in text_measures])).alias("value"))
    flags = cells.selectExpr("NULLIF(TRIM(value), '') AS text_value").selectExpr(
        "COUNT(*) AS text_measure_cells",
        "COUNT_IF(text_value IS NULL) AS blank_or_null_cells",
        "COUNT_IF(text_value IS NOT NULL AND TRY_CAST(text_value AS DOUBLE) IS NULL) AS nonnumeric_cells"
    ).first().asDict()
    print("NFHS text measure cells", flags)
assert geography["rows"] == 706, "Review the NFHS input version and row count"

NFHS geography {'rows': 706, 'incomplete_location_rows': 0, 'complete_location_pairs': 706, 'repeated_complete_pairs': 0}
NFHS measure columns by stored type {'double': 58, 'string': 49}
NFHS text measure cells {'text_measure_cells': 34594, 'blank_or_null_cells': 0, 'nonnumeric_cells': 9193}


In [0]:
# Replay the accepted local batch twice in disposable Delta tables.
from uuid import uuid4

replay_id = uuid4().hex
inputs = {
    "facilities": 10088,
    "india_post_pincode_directory": 165627,
    "nfhs_5_district_health_indicators": 706,
}
for name, expected_rows in inputs.items():
    original = f"workspace.default.bronze_{name}"
    scratch = f"workspace.default.tmp_bronze_replay_{replay_id}_{name}"
    batch = spark.sql(f"SELECT * FROM {original} VERSION AS OF 0")
    # The first write fails if the generated name is already in use.
    spark.sql(f"CREATE TABLE {scratch} USING DELTA AS SELECT * FROM {original} VERSION AS OF 0")
    try:
        first_rows = spark.table(scratch).count()
        batch.write.format("delta").mode("overwrite").saveAsTable(scratch)
        replay = spark.table(scratch)
        second_rows = replay.count()
        if batch.dtypes != replay.dtypes:
            raise ValueError(f"Replay schema mismatch for {name}")
        missing = batch.exceptAll(replay).count()
        extra = replay.exceptAll(batch).count()
        print(name, "first_rows", first_rows, "replay_rows", second_rows,
              "missing", missing, "extra", extra)
        if first_rows != expected_rows or second_rows != expected_rows or missing or extra:
            raise ValueError(f"Replay mismatch for {name}")
    finally:
        spark.sql(f"DROP TABLE {scratch}")
        print("Removed scratch table", scratch)

facilities first_rows 10088 replay_rows 10088 missing 0 extra 0
Removed scratch table workspace.default.tmp_bronze_replay_d4c720839b9847f589ed2979ab4f1e91_facilities
india_post_pincode_directory first_rows 165627 replay_rows 165627 missing 0 extra 0
Removed scratch table workspace.default.tmp_bronze_replay_d4c720839b9847f589ed2979ab4f1e91_india_post_pincode_directory
nfhs_5_district_health_indicators first_rows 706 replay_rows 706 missing 0 extra 0
Removed scratch table workspace.default.tmp_bronze_replay_d4c720839b9847f589ed2979ab4f1e91_nfhs_5_district_health_indicators


In [0]:
# One catalog table for every raw record, with explicit source and record labels.
target = "workspace.default.bronze_raw_daniel"
sources = [
    ("facilities", "facility", 10088),
    ("india_post_pincode_directory", "post_office", 165627),
    ("nfhs_5_district_health_indicators", "district_survey", 706),
]
branches = []
for dataset, kind, _ in sources:
    table = f"workspace.default.bronze_{dataset}_daniel"
    payloads = [
        f"struct(r.*) AS {field}" if field == kind else f"NULL AS {field}"
        for _, field, _ in sources
    ]
    branches.append(f"""
        SELECT '{dataset}' AS source_dataset, '{kind}' AS record_type,
               '{table}' AS source_table, 0 AS source_version, {", ".join(payloads)}
        FROM {table} VERSION AS OF 0 AS r
    """)
all_records = "\nUNION ALL\n".join(branches)
spark.sql(f"CREATE TABLE IF NOT EXISTS {target} USING DELTA AS {all_records}")
version = int(spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").first()["version"])
actual = spark.sql(f"SELECT * FROM {target} VERSION AS OF {version}")
expected = spark.sql(all_records)
if expected.dtypes != actual.dtypes:
    raise ValueError(f"Review schema of {target}")
counts = {row.record_type: row["count"] for row in actual.groupBy("record_type").count().collect()}
missing = expected.exceptAll(actual).count()
extra = actual.exceptAll(expected).count()
print(f"{target} | version={version} | rows={sum(counts.values())} | missing={missing} | extra={extra}")
print("Raw records by type", counts)
if counts != {kind: rows for _, kind, rows in sources} or missing or extra:
    raise ValueError("Review combined raw records and labels")

workspace.default.bronze_raw_daniel | version=0 | rows=176421 | missing=0 | extra=0
Raw records by type {'facility': 10088, 'post_office': 165627, 'district_survey': 706}
